In [1]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
# 0 = all messages are logged (default behavior)
# 1 = INFO messages are not printed
# 2 = INFO and WARNING messages are not printed
# 3 = INFO, WARNING, and ERROR messages are not printed

import datetime

import IPython
import IPython.display
import matplotlib as mpl
import matplotlib.pyplot as plt


from sklearn.preprocessing import StandardScaler, MinMaxScaler
from keras.models import load_model
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error

import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
#from google.colab import drive
#drive.mount('/content/gdrive')

mpl.rcParams['figure.figsize'] = (8, 6)
mpl.rcParams['axes.grid'] = False

#import warnings
# https://stackoverflow.com/questions/15777951/how-to-suppress-pandas-future-warning
#warnings.simplefilter(action='ignore', category=FutureWarning)
#warnings.simplefilter(action='ignore', category=Warning)

tf.get_logger().setLevel('ERROR')
tf.autograph.set_verbosity(0)

import logging
tf.get_logger().setLevel(logging.ERROR)

# https://stackoverflow.com/questions/65697623/tensorflow-warning-found-untraced-functions-such-as-lstm-cell-6-layer-call-and
import absl.logging
absl.logging.set_verbosity(absl.logging.ERROR)

In [2]:
# gpu_info = !nvidia-smi
# gpu_info = '\n'.join(gpu_info)
# if gpu_info.find('failed') >= 0:
#   print('Not connected to a GPU')
# else:
#   print(gpu_info)

/bin/bash: line 1: nvidia-smi: command not found


In [2]:
dfs = {}

df = pd.read_csv('/content/Station1_Revised_Final_Data.csv', sep=",", parse_dates=["Unnamed: 0"], index_col="Unnamed: 0")
dfs['Station1'] = df
# for index in range(0, 6):
#   df = pd.read_csv('Station' + str(index + 1) + '_simulated_cleaned_merged_data.csv', sep=",", parse_dates=["Unnamed: 0"], index_col="Unnamed: 0")
#   dfs['Station' + str(index + 1)] = df
#   df.index = pd.to_datetime(df.index)



In [3]:
df.describe().transpose()

,count,mean,std,min,25%,50%,75%,max
Ppt,52560.0,0.068205,8.405104e-01,-1.464373,0.0000,0.0000,0.0000,40.640000
SWC_5,52560.0,0.140254,5.154907e-02,0.004612,0.0950,0.1340,0.1830,0.309000
SWC_10,52560.0,0.157113,4.093361e-02,-0.023168,0.1230,0.1540,0.1920,0.298000
SWC_20,52560.0,0.137294,3.346671e-02,0.049357,0.1050,0.1360,0.1650,0.235000
SWC_50,52560.0,0.137377,3.578278e-02,0.076554,0.1030,0.1290,0.1700,0.249000
T_5,52560.0,22.992845,9.495459e+00,0.760000,15.5500,22.9300,30.0800,49.960000
T_10,52560.0,23.007369,8.874525e+00,2.000000,15.8000,23.1100,30.1700,44.440000
T_20,52560.0,22.948400,8.384273e+00,3.270000,15.9400,23.0800,30.0100,41.200000
T_50,52560.0,22.805075,7.007613e+00,7.880000,16.5200,22.5900,29.2300,35.400000
Tair,52560.0,14.142881,2.328434e+01,-173.200000,11.5775,19.3300,24.4200,172.476413


In [4]:
import numpy as np
import pandas as pd

def engineer_data(dfs):
    # Constants for day and year calculations
    day = 24 * 60 * 60
    year = 365.2425 * day

    for station, df in dfs.items():
        # Extract windspeed and wind direction
        wv = df['Windspeed']
        wd_rad = np.deg2rad(df['Winddirection'])  # Convert degrees to radians

        # Ensure the index is in datetime format
        df.index = pd.to_datetime(df.index)

        # Convert timestamp index to seconds
        timestamp_s = df.index.map(pd.Timestamp.timestamp) #Use the map function and pd.Timestamp.timestamp to get time in seconds since epoch.

        # Extract latitude and longitude
        lat = np.deg2rad(df['Latitude'])  # Convert latitude to radians
        lon = np.deg2rad(df['Longitude'])  # Convert longitude to radians

        # Engineer wind components
        df['Wx'] = wv * np.cos(wd_rad)
        df['Wy'] = wv * np.sin(wd_rad)

        # Engineer time-based cyclic features
        df['Day sin'] = np.sin(timestamp_s * (2 * np.pi / day))
        df['Day cos'] = np.cos(timestamp_s * (2 * np.pi / day))
        df['Year sin'] = np.sin(timestamp_s * (2 * np.pi / year))
        df['Year cos'] = np.cos(timestamp_s * (2 * np.pi / year))

        # Engineer Cartesian coordinates for latitude and longitude
        df['x_cord'] = np.cos(lat) * np.cos(lon)
        df['y_cord'] = np.cos(lat) * np.sin(lon)
        df['z_cord'] = np.sin(lat)

        # Update the DataFrame in the dictionary
        dfs[station] = df

In [5]:
engineer_data(dfs)

In [6]:
df.describe().transpose()

,count,mean,std,min,25%,50%,75%,max
Ppt,52560.0,6.820478e-02,8.405104e-01,-1.464373,0.000000,0.000000e+00,0.000000,40.640000
SWC_5,52560.0,1.402540e-01,5.154907e-02,0.004612,0.095000,1.340000e-01,0.183000,0.309000
SWC_10,52560.0,1.571126e-01,4.093361e-02,-0.023168,0.123000,1.540000e-01,0.192000,0.298000
SWC_20,52560.0,1.372937e-01,3.346671e-02,0.049357,0.105000,1.360000e-01,0.165000,0.235000
SWC_50,52560.0,1.373766e-01,3.578278e-02,0.076554,0.103000,1.290000e-01,0.170000,0.249000
T_5,52560.0,2.299284e+01,9.495459e+00,0.760000,15.550000,2.293000e+01,30.080000,49.960000
T_10,52560.0,2.300737e+01,8.874525e+00,2.000000,15.800000,2.311000e+01,30.170000,44.440000
T_20,52560.0,2.294840e+01,8.384273e+00,3.270000,15.940000,2.308000e+01,30.010000,41.200000
T_50,52560.0,2.280507e+01,7.007613e+00,7.880000,16.520000,2.259000e+01,29.230000,35.400000
Tair,52560.0,1.414288e+01,2.328434e+01,-173.200000,11.577500,1.933000e+01,24.420000,172.476413


In [7]:
def scale_data(dfs):
    for station, df in dfs.items():
        cur_df = df.copy()
        d_sin = cur_df.pop("Day sin")
        d_cos = cur_df.pop("Day cos")
        y_sin = cur_df.pop("Year sin")
        y_cos = cur_df.pop("Year cos")
        x = cur_df.pop("x_cord")
        y = cur_df.pop("y_cord")
        z = cur_df.pop("z_cord")
        scaler = MinMaxScaler()
        scaled_df = pd.DataFrame(data=scaler.fit_transform(cur_df), columns=cur_df.columns, index=cur_df.index)
        scaled_df["Day sin"] = d_sin.values
        scaled_df["Day cos"] = d_cos.values
        scaled_df["Year sin"] = y_sin.values
        scaled_df["Year cos"] = y_cos.values
        scaled_df["x_cord"] = x.values
        scaled_df["y_cord"] = y.values
        scaled_df["z_cord"] = z.values
        dfs[station] = scaled_df

In [8]:
scale_data(dfs)

/usr/local/lib/python3.10/dist-packages/sklearn/utils/_array_api.py:695: RuntimeWarning: All-NaN slice encountered
  return xp.asarray(numpy.nanmin(X, axis=axis))
/usr/local/lib/python3.10/dist-packages/sklearn/utils/_array_api.py:712: RuntimeWarning: All-NaN slice encountered
  return xp.asarray(numpy.nanmax(X, axis=axis))


In [9]:
#Definitions
TARGET_COL = "SWC_5"
TRAIN_SPLIT = 0.7
VAL_SPLIT = 0.2
WINDOW_SIZE = 24*7
SHIFT_AMT = 10
PAT = 3
MAX_EPOCHS = 25

In [10]:
def split(df, target_col=TARGET_COL, train_split = TRAIN_SPLIT, val_split = VAL_SPLIT):
  print(df.columns)
  target_idx = df.columns.get_loc(target_col)
  train_set = df[ : int(len(df) * train_split)].values
  val_set = df[int(len(df) * train_split) : int(len(df) * (train_split + val_split))].values
  test_set = df[int(len(df) * (train_split + val_split)) :].values
  return (train_set, val_set, test_set, target_idx)

In [11]:
def generate_windows(data, window_size=24, shift=24, target_idx=0):
    labels = data[:, target_idx]

    X = []
    y = []
    for i in range(len(data) - window_size - shift):
        # get window based on input width
        window = data[i : i + window_size]

        # keep track of label associated with current window
        window_label = labels[i + window_size + shift]

        X.append(window)
        y.append(window_label)

    # in new dataset, each element is a data window, and window label is single value
    return np.array(X), np.array(y)


# given data and its labels, divide the data into batches of size batch_size
def generate_batches(X, y, batch_size=32):
    # divides data into batches, drops any remainder batches smaller than specified batch size.
    # allows models to run with any batch size
    tf_dataset = tf.data.Dataset.from_tensor_slices((X, y))
    tf_dataset = tf_dataset.repeat().batch(batch_size=batch_size, drop_remainder=True)

    # tf_dataset repeats indefinitely, need to compute number of step updates to complete 1 epoch
    steps_per_epoch = len(X) // batch_size

    return (tf_dataset, steps_per_epoch)


In [12]:
def preprocess_data(df):
    # data cleaning and feature engineering
    train_set, val_set, test_set, target_idx = split(df)


    # create window data for each dataset
    X_train, y_train = generate_windows(train_set, window_size=WINDOW_SIZE, shift=SHIFT_AMT, target_idx=target_idx)
    X_val, y_val = generate_windows(val_set, window_size=WINDOW_SIZE, shift=SHIFT_AMT, target_idx=target_idx)
    X_test, y_test = generate_windows(test_set, window_size=WINDOW_SIZE, shift=SHIFT_AMT, target_idx=target_idx)

    return (X_train, y_train, X_val, y_val, X_test, y_test)

In [13]:
cur_df = dfs["Station1"]

BATCH_SIZE = 128
X_train, y_train, X_val, y_val, X_test, y_test = preprocess_data(cur_df)

# divide each dataset into batched version to feed to models
train_dataset, train_steps = generate_batches(X_train, y_train, batch_size=BATCH_SIZE)
val_dataset, val_steps = generate_batches(X_val, y_val, batch_size=BATCH_SIZE)
test_dataset, test_steps = generate_batches(X_test, y_test, batch_size=BATCH_SIZE)

Index(['Ppt', 'SWC_5', 'SWC_10', 'SWC_20', 'SWC_50', 'T_5', 'T_10', 'T_20',
       'T_50', 'Tair', 'RH', 'Windspeed', 'Winddirection', 'Srad', 'Latitude',
       'Longitude', 'Wx', 'Wy', 'Day sin', 'Day cos', 'Year sin', 'Year cos',
       'x_cord', 'y_cord', 'z_cord'],
      dtype='object')


In [22]:
def compile_and_fit(model, data, steps_per_epoch, val_data, val_steps, model_name='model/', patience=3, max_epochs=MAX_EPOCHS, batch_size=32):
    # stop running epochs if the loss stops improving for patience number of epochs
    early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=patience, mode='min')

    # store the best model on disk to be loaded later without having to re-fit
    # allows you to load models from disc
    ckpt = tf.keras.callbacks.ModelCheckpoint(model_name + ".keras", save_best_only=True)

    model.compile(loss=tf.keras.losses.MeanSquaredError(),
                  optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
                  metrics=[tf.keras.metrics.MeanAbsoluteError(), tf.keras.metrics.MeanSquaredError()])

    history = model.fit(data,
                        epochs=max_epochs,
                        callbacks=[ckpt, early_stopping],
                        validation_data=val_data,
                        validation_steps=val_steps,
                        shuffle=False,
                        batch_size=batch_size,
                        steps_per_epoch = steps_per_epoch)

    return history

In [15]:
preds = {}

def plot_single_pred(model, name, dataset, data_steps, y, batch_size=32):
    forecast = model.predict(dataset, batch_size=batch_size, steps=data_steps)
    preds[name] = forecast
    if len(forecast.shape) == 3:
        print("asd")
        forecast = forecast[:, 0, 0]
    elif len(forecast.shape) == 2:
        forecast = forecast[:, 0]

    plt.figure(figsize=(10, 6))
    plot_data = {"Predictions": forecast, "Actual": y}

    plt.plot(plot_data["Actual"])
    plt.plot(plot_data["Predictions"])

    plt.legend(("Actual", "Predictions"))

    return plot_data

In [16]:
lstm_model = tf.keras.models.Sequential([
    tf.keras.layers.LSTM(128, return_sequences=True),
    tf.keras.layers.LSTM(64, return_sequences=True),
    tf.keras.layers.LSTM(32, return_sequences=False),
    tf.keras.layers.Dense(units=32, activation='relu'),
    tf.keras.layers.Dense(units=1, activation='tanh')
])

In [17]:
bi_lstm_model = tf.keras.models.Sequential([
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(128, return_sequences=True)),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64, return_sequences=True)),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(32, return_sequences=False)),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(units=1, activation='tanh'),
])

In [18]:
loss_by_epoch = {}
val_performance = {}
performance = {}

In [25]:
# https://www.tensorflow.org/tutorials/structured_data/time_series#baseline

linear_model = tf.keras.Sequential([
    tf.keras.layers.Dense(units=1)
])


dense_model = tf.keras.Sequential([
    tf.keras.layers.Dense(units=64, activation='relu'),
    tf.keras.layers.Dense(units=32, activation='relu'),
    tf.keras.layers.Dense(units=1)
])


cnn_model = tf.keras.Sequential([
    tf.keras.layers.Conv1D(filters=32, kernel_size=5,
                      activation='relu', input_shape=X_train.shape[-2:]),
    tf.keras.layers.MaxPooling1D(pool_size=4),
    tf.keras.layers.Conv1D(filters=32, kernel_size=5, activation='relu'),
    tf.keras.layers.MaxPooling1D(pool_size=4),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(units=1)
])


rnn_model = tf.keras.Sequential([
    tf.keras.layers.SimpleRNN(128, return_sequences=True),
    tf.keras.layers.SimpleRNN(64, return_sequences=True),
    tf.keras.layers.SimpleRNN(32, return_sequences=False),
    tf.keras.layers.Dense(units=1)
])


autoregressive_model = tf.keras.Sequential([
    tf.keras.layers.Dense(units=1, input_shape=X_train.shape[-2:])
])


/usr/local/lib/python3.10/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [23]:
ckpt = tf.keras.callbacks.ModelCheckpoint("biLSTM" + ".keras", save_best_only=True) # add the .keras extension to the filepath


history = compile_and_fit(bi_lstm_model, train_dataset, train_steps, val_dataset, val_steps, batch_size=BATCH_SIZE, model_name="biLSTM", patience=PAT)
loss_by_epoch["biLSTM"] = history.history
val_performance["biLSTM"] = bi_lstm_model.evaluate(val_dataset, steps=val_steps, batch_size=BATCH_SIZE, verbose=1)
performance["biLSTM"] = bi_lstm_model.evaluate(test_dataset, steps=test_steps, batch_size=BATCH_SIZE, verbose=0)

Epoch 1/25
286/286 ━━━━━━━━━━━━━━━━━━━━ 678s 2s/step - loss: nan - mean_absolute_error: nan - mean_squared_error: nan - val_loss: nan - val_mean_absolute_error: nan - val_mean_squared_error: nan
Epoch 2/25
286/286 ━━━━━━━━━━━━━━━━━━━━ 655s 2s/step - loss: nan - mean_absolute_error: nan - mean_squared_error: nan - val_loss: nan - val_mean_absolute_error: nan - val_mean_squared_error: nan
Epoch 3/25
286/286 ━━━━━━━━━━━━━━━━━━━━ 643s 2s/step - loss: nan - mean_absolute_error: nan - mean_squared_error: nan - val_loss: nan - val_mean_absolute_error: nan - val_mean_squared_error: nan
80/80 ━━━━━━━━━━━━━━━━━━━━ 59s 740ms/step - loss: nan - mean_absolute_error: nan - mean_squared_error: nan


In [26]:
# prompt: now do the same thing with the linear model

ckpt = tf.keras.callbacks.ModelCheckpoint("linear" + ".keras", save_best_only=True) # add the .keras extension to the filepath


history = compile_and_fit(linear_model, train_dataset, train_steps, val_dataset, val_steps, batch_size=BATCH_SIZE, model_name="linear", patience=PAT)
loss_by_epoch["linear"] = history.history
val_performance["linear"] = linear_model.evaluate(val_dataset, steps=val_steps, batch_size=BATCH_SIZE, verbose=1)
performance["linear"] = linear_model.evaluate(test_dataset, steps=test_steps, batch_size=BATCH_SIZE, verbose=0)


Epoch 1/25
286/286 ━━━━━━━━━━━━━━━━━━━━ 10s 33ms/step - loss: nan - mean_absolute_error: nan - mean_squared_error: nan - val_loss: nan - val_mean_absolute_error: nan - val_mean_squared_error: nan
Epoch 2/25
286/286 ━━━━━━━━━━━━━━━━━━━━ 9s 30ms/step - loss: nan - mean_absolute_error: nan - mean_squared_error: nan - val_loss: nan - val_mean_absolute_error: nan - val_mean_squared_error: nan
Epoch 3/25
286/286 ━━━━━━━━━━━━━━━━━━━━ 9s 30ms/step - loss: nan - mean_absolute_error: nan - mean_squared_error: nan - val_loss: nan - val_mean_absolute_error: nan - val_mean_squared_error: nan
80/80 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: nan - mean_absolute_error: nan - mean_squared_error: nan


In [40]:
# prompt: why is the loss nan?

import numpy as np
# Analyze the data for potential issues causing NaN loss
print(df.isnull().sum())  # Check for missing values
print(df.describe())  # Examine data statistics for outliers or unexpected values

# Check for infinite values
print(np.any(np.isinf(df)))

# Check for potential issues in the target variable
print(df[TARGET_COL].describe())

# Examine the training data for anomalies
print('x_train shape ',X_train.shape)
print('y_train shape', y_train.shape)
print('isnan x_train',np.any(np.isnan(X_train)))
print('isnan y_train', np.any(np.isnan(y_train)))

# Check for gradient explosion issues during training
# (e.g., by examining the magnitude of gradients)

# Inspect model weights for unexpected values
# (e.g., very large or very small weights)

# Reduce learning rate
# Add gradient clipping
# Adjust the activation function

# Examine the loss function for potential issues
# (e.g., numerical instability)

# Verify the data pipeline for any errors
# (e.g., incorrect data scaling or preprocessing)


Ppt                  0
SWC_5                0
SWC_10               0
SWC_20               0
SWC_50               0
T_5                  0
T_10                 0
T_20                 0
T_50                 0
Tair                 0
RH                   0
Windspeed            0
Winddirection        0
Srad                 0
Latitude             0
Longitude            0
Wx               52560
Wy               52560
Day sin              0
Day cos              0
Year sin             0
Year cos             0
x_cord               0
y_cord               0
z_cord               0
dtype: int64
                Ppt         SWC_5        SWC_10        SWC_20        SWC_50  \
count  52560.000000  52560.000000  52560.000000  52560.000000  52560.000000   
mean       0.068205      0.140254      0.157113      0.137294      0.137377   
std        0.840510      0.051549      0.040934      0.033467      0.035783   
min       -1.464373      0.004612     -0.023168      0.049357      0.076554   
25%        0.0000

In [28]:
performance['biLSTM'][1]

nan

In [38]:
if df.isnull().values.any():
         print("NaNs found in the data!")

NaNs found in the data!


In [39]:
X_train = X_train.reshape(X_train.shape[0], -1)

X_train = pd.DataFrame(X_train)
X_train.fillna(method='ffill', inplace=True)
X_train = X_train.to_numpy()

<ipython-input-39-35333efc84dc>:4: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  X_train.fillna(method='ffill', inplace=True)
